In [2]:
import duckdb              #banco leve para análise de dados (OLAP).
import pandas as pd          #biblioteca de manipulação e análise de dados com apelido de "pd".
import os                      #permite que o código python converse com o sistema de arquivos do computador.
from datetime import datetime   #permite trabalhar com datas e horários de forma simples e direta no python.

CAMADA BRONZE

In [3]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [4]:
arquivo = 'z0019_2.csv'
data_ingestao = datetime.now()
df = pd.read_csv(f'../landing/{arquivo}', sep=';')
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-03 20:10:29.744860
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-03 20:10:29.744860
2,10003,PREGO,BT10,100,60,z0019_2.csv,2026-03-03 20:10:29.744860


In [5]:
con.execute("""
    CREATE TABLE IF NOT EXISTS bronze_produtos (
        NATBR VARCHAR,
        MAKTX VARCHAR,
        WERKS VARCHAR,
        MAINS VARCHAR,
        LABST VARCHAR,
        nome_arquivo VARCHAR,
        data_ingestao TIMESTAMP                            
    )
""")

In [21]:
con.execute("INSERT INTO bronze_produtos SELECT * FROM df")

In [6]:
resultado = con.execute("SELECT * FROM bronze_z0019").fetchdf()
resultado.head(6)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-03-03 19:09:23.900012
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-03-03 19:09:23.900012
2,10003,PREGO,BT10,100,50,z0019_1.csv,2026-03-03 19:09:23.900012
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-03 19:22:43.077335
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-03 19:22:43.077335
5,10003,PREGO,BT10,100,60,z0019_2.csv,2026-03-03 19:22:43.077335


In [24]:
con.execute("ALTER TABLE bronze_produtos RENAME TO bronze_z0019")

In [27]:
con.close()